## Read Model Weight and Run Inference

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy

In [2]:
###########################################################################
# Re-define the model architecture (must match the saved model)
###########################################################################

class FoodPreferenceANN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8):
        super(FoodPreferenceANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

In [3]:
###########################################################################
# Instantiate and load weights from .pt File
###########################################################################

model = FoodPreferenceANN(input_dim=4, hidden_dim=8)
model.load_state_dict(torch.load('ai_food_weights.pt'))
model.eval()  # Set model to evaluation mode

print("Model weights successfully loaded from ai_food_weights.pt!")

Model weights successfully loaded from ai_food_weights.pt!


In [4]:
###########################################################################
# Setup scaler (re-fitted on training feature distribution)
###########################################################################

# Note: Features are [Sour, Sweet, Salty, Spicy]
train_features = np.array([
    [1, 6, 5, 1],  # Nasi Tim Ayam
    [2, 4, 6, 7],  # Telur Balado
    [2, 3, 4, 1],  # Sayur Bening Bayam
    [2, 8, 6, 2],  # Ayam Goreng Mentega
    [1, 4, 4, 2],  # Tumis Buncis Wortel
    [1, 5, 5, 2]   # Opor Ayam
])

scaler = StandardScaler()
scaler.fit(train_features);

In [5]:
###########################################################################
# Perform inference on food items using float (original) model
###########################################################################

new_foods = [
    {"Name": "Nasi Tim Ayam",        "Sour": 1, "Sweet": 6, "Salty": 5, "Spicy": 1},
    {"Name": "Telur Balado",         "Sour": 2, "Sweet": 4, "Salty": 6, "Spicy": 7},
    {"Name": "Sayur Bening Bayam",   "Sour": 2, "Sweet": 3, "Salty": 4, "Spicy": 1},
    {"Name": "Ayam Goreng Mentega", "Sour": 2, "Sweet": 8, "Salty": 6, "Spicy": 2},
    {"Name": "Tumis Buncis Wortel",  "Sour": 1, "Sweet": 4, "Salty": 4, "Spicy": 2},
    {"Name": "Opor Ayam",            "Sour": 1, "Sweet": 5, "Salty": 5, "Spicy": 2}
]

# Extract feature matrix [Sour, Sweet, Salty, Spicy]
X_new = np.array([[f['Sour'], f['Sweet'], f['Salty'], f['Spicy']] for f in new_foods])

# Scale inputs and convert to PyTorch tensor
X_new_scaled = scaler.transform(X_new)
X_tensor = torch.tensor(X_new_scaled, dtype=torch.float32)

# Run model inference
with torch.no_grad():
    raw_logits = model(X_tensor)
    probabilities = torch.sigmoid(raw_logits)
    predictions = (probabilities >= 0.5).float()

print("\n--- Inference Results ---")
for food, prob, pred in zip(new_foods, probabilities, predictions):
    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(f"Food: {food['Name']:22s} | Prob(Like): {prob.item():.4f} | Prediction: {taste}")


--- Inference Results ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9892 | Prediction: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Prediction: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Prediction: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9961 | Prediction: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0387 | Prediction: Dislike
Food: Opor Ayam              | Prob(Like): 0.9408 | Prediction: Like


## Run Inference with Quantized Class

In [6]:
###########################################################################
# Naive matrix multiplication
###########################################################################

def manual_mmult(A, B):
    # Ensure inputs are 2D
    A_2d = A if A.ndim == 2 else A.reshape(-1, A.shape[-1])
    B_2d = B if B.ndim == 2 else B.reshape(-1, B.shape[-1])

    rows_A, cols_A = A_2d.shape
    rows_B, cols_B = B_2d.shape

    if cols_A != rows_B:
        raise ValueError(
            f"Incompatible shapes for matmul: {cols_A} != {rows_B}"
        )

    # Initialize 2D output array (using int32 for FPGA accumulation)
    result_2d = np.zeros((rows_A, cols_B), dtype=np.int32)

    # Standard matrix multiplication loops: (M x K) @ (K x N) -> (M x N)
    for i in range(rows_A):
        for j in range(cols_B):
            acc = 0
            for k in range(cols_A):
                acc += int(A_2d[i, k]) * int(B_2d[k, j])
            result_2d[i, j] = acc

    return result_2d

In [7]:
###########################################################################
# Quantized linear class
###########################################################################

class NumpyQuantizedLinear(nn.Module):

    def __init__(self, original_layer, act_scale):
        super().__init__()
        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        # Layer-specific activation scale
        self.act_scale = act_scale

        # Layer-specific weight scale
        weights_float = original_layer.weight.data.detach().cpu().numpy()
        self.weight_scale = np.max(np.abs(weights_float)) / 127
        self.weights_int8 = np.round(weights_float / self.weight_scale).astype(
            np.int8
        )

    def forward(self, x):
        # --- PRE-PROCESSING (Simulating CPU -> FPGA Transfer) ---
        # Scale and convert input to int8
        x_np = x.detach().cpu().numpy()
        x_int8 = np.clip(x_np / self.act_scale * 127, -128, 127).astype(np.int8)

        # --- DEBUG PRINT ---
        # x1 is the input activations, x2 is the transposed weights
        x1_shape = x_int8.shape
        x2_shape = self.weights_int8.T.shape
        print(f"DEBUG: matmul x1 {x1_shape} @ x2 {x2_shape}")
        # print(f"    x1 = {x_int8}")
        # print(f"    x2 = {self.weights_int8.T}")
        # print(f"==============================================")

        # --- THE "SIMULATED HARDWARE" CORE ---
        # Perform matrix multiplication in INT32 to prevent overflow
        # (Just like an FPGA accumulator)
        # out_int32 = np.matmul(x_int8.astype(np.int32), self.weights_int8.T.astype(np.int32))
        out_int32 = manual_mmult(x_int8.astype(np.int32), self.weights_int8.T.astype(np.int32))

        # --- POST-PROCESSING (Simulating FPGA -> CPU Transfer) ---
        # Convert back to float using both scales
        total_scale = (self.act_scale / 127) * self.weight_scale
        out_float = out_int32.astype(np.float32) * total_scale

        return torch.from_numpy(out_float).to(x.device)

In [9]:
###########################################################################
# Utility functions to modify the model
###########################################################################

def integrate_numpy_quantization(model, act_scale):
    """
    Recursively replaces all nn.Linear layers with NumpyQuantizedLinear.
    """
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # 1. Create the new quantized layer using the original layer's data
            quantized_layer = NumpyQuantizedLinear(module, act_scale)

            # 2. Replace the old layer with the new one in the model
            setattr(model, name, quantized_layer)
            print(f"Replaced layer: {name}")
        else:
            # 3. If it's a container (like Sequential or MultiheadAttention),
            # recurse into its children
            integrate_numpy_quantization(module, act_scale)

def get_calibration_scale(model, calibration_loader):
    model.eval()
    max_values = []

    with torch.no_grad():
        for batch in calibration_loader:
            # 1. Unpack the Tensor from the DataLoader list/tuple
            x = batch[0] if isinstance(batch, (list, tuple)) else batch

            # 2. Get model outputs (single Tensor output)
            logits = model(x)

            # 3. Track maximum activation magnitude across the batch
            max_values.append(torch.max(torch.abs(logits)).item())

    # Return the average maximum activation
    return sum(max_values) / len(max_values)

In [10]:
###########################################################################
# Get activation scale
###########################################################################

# Transform the raw training features using the fitted scaler
X_scaled = scaler.transform(train_features)
# Convert the scaled NumPy array into a PyTorch FloatTensor
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
# Create TensorDataset and wrap it in a DataLoader
# (batch_size can be set to your preferred batch size, e.g., 2 or 6)
calibration_dataset = TensorDataset(X_tensor)
calibration_loader = DataLoader(
    calibration_dataset, batch_size=2, shuffle=False
)

# Find the activation scale from floating-point calibration runs
act_scale = get_calibration_scale(model, calibration_loader)
print(f"Calculated Activation Scale: {act_scale}")

Calculated Activation Scale: 5.428282737731934


In [11]:
###########################################################################
# Replace the nn.Linear class
###########################################################################

# Create an isolated deep copy for quantization
model_quantized = copy.deepcopy(model)

# Verify layer before modification
print("Before (Original Model Layer1): ", model.layer1)
print("Before (Quantized Copy Layer1): ", model_quantized.layer1)

# Apply quantization ONLY to the copied model
integrate_numpy_quantization(model_quantized, act_scale)

print("\n--- After Quantization ---")
# Verify layer after modification
print("Original model remains float:   ", model.layer1)
print("Quantized model replaced layer: ", model_quantized.layer1)

Before (Original Model Layer1):  Linear(in_features=4, out_features=8, bias=True)
Before (Quantized Copy Layer1):  Linear(in_features=4, out_features=8, bias=True)
Replaced layer: layer1
Replaced layer: layer2

--- After Quantization ---
Original model remains float:    Linear(in_features=4, out_features=8, bias=True)
Quantized model replaced layer:  NumpyQuantizedLinear()


In [12]:
###########################################################################
# Perform inference on food items using quantized model
###########################################################################

new_foods = [
    {"Name": "Nasi Tim Ayam", "Sour": 1, "Sweet": 6, "Salty": 5, "Spicy": 1},
    {"Name": "Telur Balado", "Sour": 2, "Sweet": 4, "Salty": 6, "Spicy": 7},
    {"Name": "Sayur Bening Bayam", "Sour": 2, "Sweet": 3, "Salty": 4, "Spicy": 1},
    {"Name": "Ayam Goreng Mentega", "Sour": 2, "Sweet": 8, "Salty": 6, "Spicy": 2},
    {"Name": "Tumis Buncis Wortel", "Sour": 1, "Sweet": 4, "Salty": 4, "Spicy": 2},
    {"Name": "Opor Ayam", "Sour": 1, "Sweet": 5, "Salty": 5, "Spicy": 2},
]

# Extract and scale inputs
X_new = np.array(
    [[f["Sour"], f["Sweet"], f["Salty"], f["Spicy"]] for f in new_foods]
)
X_new_scaled = scaler.transform(X_new)
X_tensor = torch.tensor(X_new_scaled, dtype=torch.float32)

# Set quantized model to evaluation mode
model_quantized.eval()

with torch.no_grad():
    # Run quantized model inference
    raw_logits_quant = model_quantized(X_tensor)

    # Convert NumPy output to PyTorch Tensor if your custom layer returns NumPy arrays
    if isinstance(raw_logits_quant, np.ndarray):
        raw_logits_quant = torch.from_numpy(raw_logits_quant).float()

    probabilities_quant = torch.sigmoid(raw_logits_quant)
    predictions_quant = (probabilities_quant >= 0.5).float()

# Print Quantized Model Results
print("\n--- Quantized Model Inference Results ---")
for food, prob, pred in zip(new_foods, probabilities_quant, predictions_quant):
    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(
        f"Food: {food['Name']:22s} | Quant Prob(Like): {prob.item():.4f} | Prediction: {taste}"
    )

DEBUG: matmul x1 (6, 4) @ x2 (4, 8)
DEBUG: matmul x1 (6, 8) @ x2 (8, 1)

--- Quantized Model Inference Results ---
Food: Nasi Tim Ayam          | Quant Prob(Like): 0.9144 | Prediction: Like
Food: Telur Balado           | Quant Prob(Like): 0.0150 | Prediction: Dislike
Food: Sayur Bening Bayam     | Quant Prob(Like): 0.0010 | Prediction: Dislike
Food: Ayam Goreng Mentega    | Quant Prob(Like): 0.9704 | Prediction: Like
Food: Tumis Buncis Wortel    | Quant Prob(Like): 0.0284 | Prediction: Dislike
Food: Opor Ayam              | Quant Prob(Like): 0.7157 | Prediction: Like


In [13]:
###########################################################################
# Quantization error calculation
###########################################################################

# Set quantized model to evaluation mode
model_quantized.eval()

with torch.no_grad():
    # Run quantized model inference
    raw_logits_quant = model_quantized(X_tensor)

    # Ensure output is a PyTorch Tensor
    if isinstance(raw_logits_quant, np.ndarray):
        raw_logits_quant = torch.from_numpy(raw_logits_quant).float()

    probabilities_quant = torch.sigmoid(raw_logits_quant)
    predictions_quant = (probabilities_quant >= 0.5).float()

print("\n--- Quantized Model Inference & Probability Comparison ---")
print(
    f"{'Food Name':22s} | {'Float Prob':>10s} | {'Quant Prob':>10s} | {'Abs Error':>10s} | {'Prediction':>10s}"
)
print("-" * 72)

errors = []
for food, prob_orig, prob_q, pred in zip(
    new_foods, probabilities, probabilities_quant, predictions_quant
):
    orig_val = prob_orig.item()
    quant_val = prob_q.item()
    abs_err = abs(orig_val - quant_val)
    errors.append(abs_err)

    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(
        f"{food['Name']:22s} | {orig_val:10.4f} | {quant_val:10.4f} | {abs_err:10.4f} | {taste:>10s}"
    )

mae = np.mean(errors)
max_err = np.max(errors)
print("-" * 72)
print(f"Mean Absolute Error (MAE):     {mae:.6f}")
print(f"Max Absolute Error:            {max_err:.6f}")

DEBUG: matmul x1 (6, 4) @ x2 (4, 8)
DEBUG: matmul x1 (6, 8) @ x2 (8, 1)

--- Quantized Model Inference & Probability Comparison ---
Food Name              | Float Prob | Quant Prob |  Abs Error | Prediction
------------------------------------------------------------------------
Nasi Tim Ayam          |     0.9892 |     0.9144 |     0.0748 |       Like
Telur Balado           |     0.0067 |     0.0150 |     0.0083 |    Dislike
Sayur Bening Bayam     |     0.0003 |     0.0010 |     0.0007 |    Dislike
Ayam Goreng Mentega    |     0.9961 |     0.9704 |     0.0257 |       Like
Tumis Buncis Wortel    |     0.0387 |     0.0284 |     0.0103 |    Dislike
Opor Ayam              |     0.9408 |     0.7157 |     0.2251 |       Like
------------------------------------------------------------------------
Mean Absolute Error (MAE):     0.057493
Max Absolute Error:            0.225126
